# RAG Evaluation with the Galileo Python SDK

This notebook keeps the retrieval traces, metrics, dataset creation, and experiment calls inline so you can see the end-to-end RAG evaluation flow directly in code cells.


In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')


In [ ]:
from galileo import GalileoLogger, GalileoMetrics, Message, MessageRole, galileo_context
from galileo.config import GalileoPythonConfig
from galileo.datasets import create_dataset, delete_dataset
from galileo.experiments import PromptRunSettings, PromptTemplate, run_experiment
from galileo.log_streams import create_log_stream, enable_metrics
from galileo.projects import delete_project

PROJECT_NAME = 'GalileoEval_S2_RAG'
LOG_STREAM_NAME = 'rag-evaluation'
DATASET_NAME = 'rag-eval-dataset'
MODEL = 'gpt-4o'
dataset_id = None

KNOWLEDGE_BASE = {
    'What is Galileo?': [
        'Galileo is an AI observability platform that helps teams evaluate, monitor, and improve LLM applications.',
        'Founded in 2021, Galileo provides metrics for RAG, agentic, and conversational AI systems.',
        "Galileo's Luna-2 model enables cost-effective evaluation at scale.",
    ],
    'How does RAG work?': [
        'RAG combines information retrieval with text generation.',
        'A retriever searches a knowledge base for relevant documents.',
        'The retrieved documents are passed as context to an LLM.',
    ],
    'What metrics does Galileo offer?': [
        'Galileo offers RAG metrics including context adherence, chunk relevance, and completeness.',
        'Agentic metrics include tool selection quality, action advancement, and agent efficiency.',
        'Safety metrics cover PII detection, toxicity, prompt injection, and bias detection.',
    ],
}


## 1. Initialize Galileo

Initialize the Galileo context, make sure the log stream exists, and print the console links if Galileo returns IDs.


In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

try:
    create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)
except Exception:
    pass

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    print('Galileo context initialized')


## 2. Log RAG Traces


In [ ]:
logger = GalileoLogger(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

chunks = KNOWLEDGE_BASE['What is Galileo?']
answer = 'Galileo is an AI observability platform founded in 2021 that helps teams evaluate and monitor LLM applications.'
logger.start_trace(input='What is Galileo?')
logger.add_retriever_span(input='What is Galileo?', output=chunks, duration_ns=50_000_000)
logger.add_llm_span(
    input=[
        Message(role=MessageRole.system, content='Answer from this context:\n' + '\n'.join(chunks)),
        Message(role=MessageRole.user, content='What is Galileo?'),
    ],
    output=Message(role=MessageRole.assistant, content=answer),
    model=MODEL,
    num_input_tokens=len(' '.join(chunks).split()),
    num_output_tokens=len(answer.split()),
    duration_ns=200_000_000,
)
logger.conclude(output=answer)

chunks = KNOWLEDGE_BASE['How does RAG work?']
answer = 'RAG combines retrieval and generation: a retriever finds relevant documents, then an LLM answers with that context.'
logger.start_trace(input='How does RAG work?')
logger.add_retriever_span(input='How does RAG work?', output=chunks, duration_ns=50_000_000)
logger.add_llm_span(
    input=[
        Message(role=MessageRole.system, content='Answer from this context:\n' + '\n'.join(chunks)),
        Message(role=MessageRole.user, content='How does RAG work?'),
    ],
    output=Message(role=MessageRole.assistant, content=answer),
    model=MODEL,
    num_input_tokens=len(' '.join(chunks).split()),
    num_output_tokens=len(answer.split()),
    duration_ns=200_000_000,
)
logger.conclude(output=answer)

chunks = KNOWLEDGE_BASE['What metrics does Galileo offer?']
answer = 'Galileo offers a free tier with unlimited API calls and supports over 200 programming languages.'
logger.start_trace(input='What metrics does Galileo offer?')
logger.add_retriever_span(input='What metrics does Galileo offer?', output=chunks, duration_ns=50_000_000)
logger.add_llm_span(
    input=[
        Message(role=MessageRole.system, content='Answer from this context:\n' + '\n'.join(chunks)),
        Message(role=MessageRole.user, content='What metrics does Galileo offer?'),
    ],
    output=Message(role=MessageRole.assistant, content=answer),
    model=MODEL,
    num_input_tokens=len(' '.join(chunks).split()),
    num_output_tokens=len(answer.split()),
    duration_ns=200_000_000,
)
logger.conclude(output=answer)

logger.flush()
'Logged 3 RAG traces'


## 3. Enable Core RAG Metrics


In [ ]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[
        GalileoMetrics.context_adherence,
        GalileoMetrics.chunk_relevance,
        GalileoMetrics.completeness,
        GalileoMetrics.context_relevance,
    ],
)


## 4. Enable Additional RAG Metrics


In [ ]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[
        GalileoMetrics.context_precision,
        GalileoMetrics.precision_at_k,
        GalileoMetrics.chunk_attribution_utilization,
    ],
)


## 5. Create an Evaluation Dataset


In [ ]:
ds = create_dataset(
    name=DATASET_NAME,
    content=[
        {'input': 'What is Galileo?', 'expected_output': 'Galileo is an AI observability platform for evaluating LLM applications.'},
        {'input': 'How does RAG work?', 'expected_output': 'RAG combines retrieval of relevant documents with LLM generation.'},
        {'input': 'What metrics does Galileo offer?', 'expected_output': 'Galileo offers RAG metrics, agentic metrics, and safety metrics.'},
    ],
    project_name=PROJECT_NAME,
)
dataset_id = ds.id

ds.add_rows([
    {
        'input': 'What is context adherence?',
        'expected_output': 'Context adherence measures how well an LLM response aligns with the provided context.',
    },
])
str(dataset_id)


## 6. Run a Prompt Experiment


In [ ]:
prompt = PromptTemplate(
    messages=[
        {'role': 'system', 'content': 'You are a knowledgeable AI assistant. Answer accurately and concisely.'},
        {'role': 'user', 'content': '{{input}}'},
    ]
)

run_experiment(
    experiment_name='rag-prompt-comparison-v1',
    dataset=DATASET_NAME,
    prompt_template=prompt,
    metrics=[
        GalileoMetrics.correctness,
        GalileoMetrics.instruction_adherence,
        GalileoMetrics.ground_truth_adherence,
    ],
    prompt_settings=PromptRunSettings(model_alias='GPT-4o Mini', max_tokens=150),
    project=PROJECT_NAME,
)
'Experiment complete'


## 7. Enable Ground Truth Adherence


In [ ]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[GalileoMetrics.ground_truth_adherence],
)


## 8. Clean Up the Dataset and Project


In [ ]:
try:
    if dataset_id is not None:
        delete_dataset(id=str(dataset_id))
    else:
        delete_dataset(name=DATASET_NAME, project_name=PROJECT_NAME)
except Exception:
    pass

delete_project(name=PROJECT_NAME)
